In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

cars_schema =  StructType(
    [
        StructField("Branch_ID", StringType(), True),
        StructField("Dealer_ID", StringType(), True),
        StructField("Model_ID", StringType(), True),
        StructField("Revenue", StringType(), True),
        StructField("Units_Sold", IntegerType(), True),
        StructField("Date_ID", StringType(), True),
        StructField("Day", IntegerType(), True),
        StructField("Month", IntegerType(), True),
        StructField("Year", IntegerType(), True),
        StructField("BranchName", StringType(), True),
        StructField("DealerName", StringType(), True),
        StructField("Product_Name", StringType(), True),
        StructField("_rescued_data", StringType(), True)]
)

df = spark.readStream\
            .format("cloudFiles")\
            .option("cloudFiles.format", "csv")\
            .option("header", "true")\
            .schema(cars_schema)\
            .load("abfss://raw@storagesensorstudio.dfs.core.windows.net/Sales")

df = df.withColumn("filePath", col("_metadata.file_path"))\
    .withColumn("fileTimestamp", col("_metadata.file_modification_time"))\
    .withColumn("fileSize", col("_metadata.file_size"))\
    .withColumn("fileName", col("_metadata.file_name"))


df.writeStream\
    .format("delta")\
    .option("checkpointLocation", "abfss://bronze@storagesensorstudio.dfs.core.windows.net/checkpoints/")\
    .trigger(availableNow=True)\
    .option("path", "abfss://bronze@storagesensorstudio.dfs.core.windows.net/data")\
    .table("sensor_catalog.bronze.cars_sale")

    

In [0]:
%sql
select * from sensor_catalog.bronze.cars_sale limit 10;